# InsightForge — End-to-End Demo Notebook

This notebook walks through the full pipeline:
1. Load and inspect the sales dataset
2. Compute the summary statistics
3. Build the FAISS knowledge base
4. Run sample RAG queries
5. Run the QAEvalChain evaluation

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Load Data

In [ ]:
from src.data_loader import load_data, compute_summary

df = load_data('../data/sales_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 2. Summary Statistics

In [ ]:
summary = compute_summary(df)

print(f"Total Sales:          ${summary['total_sales']:,.2f}")
print(f"Total Orders:         {summary['total_orders']:,}")
print(f"Avg Order Value:      ${summary['avg_order_value']:,.2f}")
print(f"Avg Satisfaction:     {summary['avg_satisfaction']}/5.0")
print(f"Date Range:           {summary['date_range']}")

In [ ]:
import pandas as pd

pd.DataFrame(summary['sales_by_product'].items(), columns=['Product', 'Sales']).set_index('Product')

## 3. Build Knowledge Base

In [ ]:
from src.knowledge_base import build_knowledge_base

vectorstore = build_knowledge_base(force_rebuild=True)
print('Knowledge base ready.')

In [ ]:
# Inspect a retrieval
results = vectorstore.similarity_search('customer satisfaction by product', k=2)
for r in results:
    print(f"[{r.metadata['topic']}]")
    print(r.page_content[:300])
    print('---')

## 4. RAG Chain — Sample Queries

In [ ]:
from src.rag_system import build_rag_chain

chain = build_rag_chain()

questions = [
    'What is the total sales revenue across all orders?',
    'Which product has the highest average customer satisfaction?',
    'How do the North and South regions compare in total sales?',
    'Which age group places the most orders?',
]

for q in questions:
    result = chain.invoke({'question': q})
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print()

## 5. QAEvalChain Evaluation

In [ ]:
from src.evaluator import run_evaluation

results = run_evaluation(chain)

for r in results:
    print(f"Q:  {r['question']}")
    print(f"Grade: {r['grade']}")
    print()

## 6. Quick Visualisation

In [ ]:
from src.visualizations import sales_by_product, sales_by_region, satisfaction_by_product

sales_by_product(df).show()
sales_by_region(df).show()
satisfaction_by_product(df).show()